# Complete Guide to RNN (Recurrent Neural Networks)

## What is RNN?

RNN stands for **Recurrent Neural Network**. It's a neural network designed to process **sequences** (data that comes in order).

```
Regular Neural Network (what we learned before):
Input → Network → Output
Done!
No memory of past inputs

RNN (Recurrent):
Input₁ → Network → Output₁
  ↓ (remembers this)
Input₂ → Network → Output₂
  ↓ (remembers previous)
Input₃ → Network → Output₃
  ↓ (remembers all previous)
...

Key difference: RNN has MEMORY of previous inputs!
```

### Real-World Examples Where RNN Shines

```
Predicting Next Word in Sentence:
"The cat is sitting on the ___"
Input sequence: "The", "cat", "is", "sitting", "on", "the"
RNN remembers all previous words
Predicts next: "mat" or "floor"

Time Series Prediction:
Stock prices: [100, 102, 101, 103, 105, ?]
RNN remembers all previous prices
Predicts next price

Language Translation:
English: "Good morning"
RNN processes: "Good" → remembers → "morning" → remembers
Translates to: "Buenos días"

Speech Recognition:
Audio sequence: [sound₁, sound₂, sound₃, ...]
RNN remembers context
Transcribes to text

Sentiment Analysis:
Review: "This movie was amazing but the ending was terrible"
RNN reads word by word
Remembers context
Predicts: Negative (because "terrible" is strong)
```

---

## Why Regular Neural Networks Fail on Sequences

### The Problem

```
Sentence: "The cat sat on the mat"

Regular Neural Network approach:
- Takes ALL words at once
- No sense of order
- Doesn't know "The" comes before "cat"
- Treats sequence like a bag of words

Result:
"The cat sat on the mat" → Jumbled prediction
"sat cat The on mat the" → Same result!
(Network can't tell difference!)

RNN approach:
- Processes one word at a time
- Remembers previous words
- Knows "The" came before "cat"
- Respects sequence order

Result:
Correct prediction considering order!
```

### Sequential Data Types

```
Text (Sequence):
"Hello world" → H, e, l, l, o, space, w, o, r, l, d
Position matters!
Order matters!

Time Series (Sequence):
Stock prices: 100, 102, 101, 103, 105
Today depends on yesterday
Order crucial!

Audio (Sequence):
Sound waves: sample₁, sample₂, sample₃, ...
Samples in order create meaning
Scramble = noise!

Video (Sequence):
Frames: frame₁, frame₂, frame₃, ...
Consecutive frames show movement
Order essential!

RNN handles ALL these because it respects order!
```

---

## How RNN Works - The Core Concept

### RNN Cell (Single Time Step)

```
Standard RNN Cell:

Current Input: x_t
Previous Memory: h_(t-1)

      ┌─────────────┐
x_t ──┤             ├── h_t (output/hidden state)
      │  RNN Cell   ├── Used as memory for next step
h_(t-1) ──┤             │
      └─────────────┘

Simple math:
h_t = activation(W × [h_(t-1), x_t] + b)
      └─────────────┬──────────────┘
                  Combines previous memory 
                  + new input

Output: prediction_t = output_layer(h_t)
```

### Processing a Sequence Step by Step

```
Sentence: "The cat sat"

STEP 1: Process "The"
Input: "The"
Previous memory: 0 (empty, first word)

┌──────────────────┐
│ "The" + empty    │
│ → output: "The"  │
│ → memory: ???    │ (captures meaning of "The")
└──────────────────┘
         ↓
      Memory1

STEP 2: Process "cat"
Input: "cat"
Previous memory: Memory1 (from "The")

┌──────────────────┐
│ "cat" + Memory1  │
│ → output: "cat"  │
│ → memory: ???    │ (captures "The cat")
└──────────────────┘
         ↓
      Memory2

STEP 3: Process "sat"
Input: "sat"
Previous memory: Memory2 (from "The cat")

┌──────────────────┐
│ "sat" + Memory2  │
│ → output: "sat"  │
│ → memory: ???    │ (captures "The cat sat")
└──────────────────┘
         ↓
    Final Memory

Using Final Memory:
Predict next word: "on" (because network learned this pattern)
```

### Visual RNN Unfolding

```
Regular Neural Network (doesn't process sequence):
x₁, x₂, x₃ → [All at once] → y

RNN (processes sequence step by step):

Time Step 1:        Time Step 2:        Time Step 3:
    x₁                  x₂                  x₃
     ↓                  ↓                   ↓
  ┌─────┐             ┌─────┐            ┌─────┐
  │RNN  │ h₁          │RNN  │ h₂         │RNN  │ h₃
  │Cell ├─────→ y₁    │Cell ├─────→ y₂   │Cell ├─────→ y₃
  │     ├─────┐       │     ├─────┐      │     │
  └─────┘    memory1  └─────┘   memory2  └─────┘
               ↓                   ↓
            (feedback)          (feedback)

Same RNN cell used 3 times!
But with different memories each time
```

---

## Understanding Hidden State (Memory)

### What is Hidden State?

Hidden state is a **vector of numbers** that stores information about all previous inputs.

```
After seeing: "The"
h₁ = [0.5, -0.2, 0.8, -0.1, ...]
(Captures: "We saw an article")

After seeing: "The cat"
h₂ = [0.7, -0.3, 0.9, -0.2, ...]
(Captures: "We saw an article about a cat")

After seeing: "The cat sat"
h₃ = [0.8, -0.4, 0.95, -0.3, ...]
(Captures: "We saw an article about a cat sitting")

The numbers change based on each new word!
The vector GROWS in understanding!
```

### Hidden State Size

```
Hidden state dimension: 128 (typical)

h_t = [num₁, num₂, num₃, ..., num₁₂₈]

What do these numbers represent?
They're learned during training!
Could be:
- num₁: "Is this positive?"
- num₂: "Is this past tense?"
- num₃: "Is subject singular?"
- num₄: "Is this about animals?"
... (network learns what each represents)

Larger hidden state:
More capacity to remember
But harder to train

Typical sizes: 64, 128, 256, 512
```

---

## RNN Formula Explained Simply

### Mathematical Formula

```
Hidden state update:
h_t = tanh(W_h × [h_(t-1), x_t] + b_h)

Breaking it down:

h_(t-1): Previous hidden state (memory)
         Shape: (hidden_size,)
         Example: (128,) = 128 numbers

x_t: Current input
     Shape: (input_size,)
     Example: (50,) = 50 numbers (word embedding)

[h_(t-1), x_t]: Concatenate (join) both
                Shape: (128 + 50,) = (178,)

W_h: Weight matrix
     Shape: (178, 128)
     Transforms combined input back to hidden size

W_h × [h_(t-1), x_t]: Matrix multiplication
                      Result shape: (128,)

+ b_h: Add bias
       Shape: (128,)

tanh(...): Activation function
           Squashes values to [-1, 1]

Result: h_t = new hidden state (128 numbers)
```

### Output Formula

```
Output at each time step:
y_t = softmax(W_y × h_t + b_y)

W_y: Output weight matrix
     Transforms hidden state to output

Example for predicting next word:
h_t shape: (128,)
W_y shape: (128, 10000)  [vocab size = 10000 words]

Result: y_t shape (10000,)
Each number = probability of each word
```

---

## Complete RNN Example - Predicting Next Word

### Setup

```
Task: Predict next word in sentence

Sentence: "The cat sat on the mat"

Vocabulary: 50 unique words
We'll predict which word comes next

Hidden size: 4 (small, for simplicity)
```

### Step 1: Convert Words to Embeddings

```
Words aren't numbers, need to convert

Word embedding (representation):
"The" → [0.1, -0.2, 0.5, 0.3]
"cat" → [0.2, 0.1, -0.3, 0.4]
"sat" → [-0.1, 0.3, 0.2, -0.2]
"on" → [0.4, -0.1, 0.1, 0.3]
"mat" → [0.3, 0.2, -0.1, 0.4]

(These are learned during training)
```

### Step 2: Time Step 1 - Input "The"

```
Input: "The" → [0.1, -0.2, 0.5, 0.3]
Previous hidden state: h₀ = [0, 0, 0, 0] (start, empty memory)

Combine: [0, 0, 0, 0, 0.1, -0.2, 0.5, 0.3]

Weights multiply and transform:
h₁ = tanh(transform)
h₁ = [0.2, -0.1, 0.3, 0.1]

Output prediction:
y₁ = softmax(W_y × h₁)
y₁ = [0.15, 0.08, 0.12, 0.25, 0.10, ...] (50 probabilities)

Predicted next word: Position 3 (highest probability)
But we want "cat", so network will adjust weights
```

### Step 3: Time Step 2 - Input "cat"

```
Input: "cat" → [0.2, 0.1, -0.3, 0.4]
Previous hidden state: h₁ = [0.2, -0.1, 0.3, 0.1]

Combine: [0.2, -0.1, 0.3, 0.1, 0.2, 0.1, -0.3, 0.4]

Hidden state update:
h₂ = tanh(transform)
h₂ = [0.3, 0.0, 0.4, 0.2]
(Changed because of new input "cat"!)

Output prediction:
y₂ = softmax(W_y × h₂)
y₂ = [0.05, 0.02, 0.25, 0.15, 0.10, ...] (50 probabilities)

Predicted next word: Position 2 (highest probability)
But we want "sat", so network will adjust again
```

### Step 4: Time Step 3 - Input "sat"

```
Input: "sat" → [-0.1, 0.3, 0.2, -0.2]
Previous hidden state: h₂ = [0.3, 0.0, 0.4, 0.2]

Combine: [0.3, 0.0, 0.4, 0.2, -0.1, 0.3, 0.2, -0.2]

Hidden state update:
h₃ = tanh(transform)
h₃ = [0.35, 0.1, 0.45, 0.15]
(Changed again! Now remembers "The cat sat")

Output prediction:
y₃ = softmax(W_y × h₃)
y₃ = [0.02, 0.01, 0.05, 0.50, 0.10, ...] (50 probabilities)

Predicted next word: Position 3 (highest probability)
Should be "on" - if correct, network reinforces this pattern!
```

### Step 5: Continue

```
Step 4: Input "on" → h₄ = [...]
        Predict: "the"
        
Step 5: Input "the" → h₅ = [...]
        Predict: "mat"

Final: Network learned the pattern!
"The cat sat on the mat"

Now for new sentence: "The dog ..."
Network can apply learned patterns about articles, nouns, verbs!
```

---

## RNN Shape Tracking (Like CNN)

### Input/Output Shapes

```
SEQUENCE PROCESSING:

Input sequence: ["The", "cat", "sat"] (3 time steps)
Embedding size: 50 (each word becomes 50 numbers)

Input shape for each time step:
(batch_size, embedding_size) = (32, 50)
32 sentences, each word is 50-dimensional

After RNN with hidden_size = 128:
Output shape for each time step:
(batch_size, hidden_size) = (32, 128)

All time steps stacked:
(batch_size, sequence_length, hidden_size)
(32, 3, 128)

Meaning:
32 sentences
Each with 3 words
Each word produces 128-dimensional output
```

### Complete Shape Flow

```
Input: Sentences with sequence length 5
       Each word embedded as 50 dimensions
       Batch size: 32

Shape: (32, 5, 50)
       32 = batch size
       5 = sequence length
       50 = embedding dimension

       ↓ Process through RNN (hidden_size=128)

RNN output for each time step: (32, 128)
All time steps: (32, 5, 128)

       ↓ Take last time step only

Final hidden state: (32, 128)

       ↓ Dense layer (output_size=10000 for vocab)

Prediction shape: (32, 10000)
32 = batch size
10000 = vocab size (probability for each word)

       ↓ Softmax

Output: Probability distribution over vocab
        For each sentence in batch
        [0.02, 0.01, 0.25, ..., 0.03]
```

---

## RNN Variants - LSTM and GRU

### Problem with Basic RNN

```
Vanishing Gradient Problem:

Short sequence: "The cat sat"
RNN can learn patterns

Long sequence: "In the country of France, there is a famous river 
                 called the Seine, which flows through Paris. The 
                 architecture includes..." (100+ words)

RNN struggles because:
- Gradients get smaller through time
- Information from early words (France) fades away
- Network forgets the beginning

Basic RNN memory is SHORT-TERM only!
```

### LSTM (Long Short-Term Memory)

```
Solution: Add special memory cells

Basic RNN: One hidden state (short-term memory)

LSTM: 
- Hidden state h_t (short-term)
- Cell state c_t (long-term memory)

Cell state travels through time unchanged!
Special gates control what to remember:

1. Forget gate: "Forget irrelevant information"
2. Input gate: "Remember important new info"
3. Output gate: "What should we output?"

Advantage:
Can maintain long-term dependencies!
Remembers from very early in sequence

Example:
"In the country of France ... The capital is Paris"
LSTM remembers "France" until needed
Knows Paris is capital of France, not other countries

Formula (simplified):
c_t = forget_gate × c_(t-1) + input_gate × new_info
h_t = output_gate × tanh(c_t)

Cell state (c_t) acts as long-term memory highway!
```

### GRU (Gated Recurrent Unit)

```
Simpler than LSTM but similar idea

GRU has:
- Reset gate: "Forget some past"
- Update gate: "Update hidden state"

Fewer gates = simpler = faster training
Similar performance to LSTM in many cases

Usually LSTM preferred for:
- Long sequences (100+ words)
- Need to remember far-back info

Usually GRU preferred for:
- Shorter sequences
- Computational efficiency
- Faster training
```

### Comparison

```
Basic RNN:
✓ Simple
✓ Fast
✗ Forgets long-term info
✗ Vanishing gradient problem

LSTM:
✓ Long-term memory
✓ Handles long sequences
✓ Stable gradients
✗ Slower (more computation)
✗ More parameters to train

GRU:
✓ Good balance
✓ Faster than LSTM
✓ Handles long sequences
✗ Slightly less capacity than LSTM

Practical:
- Default: LSTM (proven, reliable)
- Large datasets: Try GRU (faster)
- Short sequences: Basic RNN (simpler)
```

---

## RNN Applications with Examples

### 1. Text Generation

```
Task: Generate realistic text

Training:
Input: "The quick brown"
Output: "fox"

Input: "The quick brown fox"
Output: "jumps"

Input: "The quick brown fox jumps"
Output: "over"

Network learns: Predicts next word based on previous words

After training, generate text:
1. Start: "The"
2. Predict next: "quick" (from RNN)
3. Use "The quick" to predict: "brown"
4. Continue: "The quick brown fox jumps over..."

Can generate full paragraphs!
```

### 2. Machine Translation

```
Task: Translate English to French

Architecture: Encoder-Decoder

Encoder RNN:
"Hello world" → 
"Hello" → h₁
"world" → h₂ (remembers "Hello")
Final state: Contains meaning of "Hello world"

Decoder RNN:
Takes final state of encoder
Generates French word by word:
h₂ → "Bonjour"
h₂ → "Bonjour" + predict → "monde"

Result: "Bonjour le monde"
```

### 3. Sentiment Analysis

```
Task: Determine if review positive or negative

Input: "This movie was amazing but the ending was terrible"

Process word by word:
"This" → h₁
"movie" → h₂ (remembers "This movie")
"was" → h₃
"amazing" → h₄ (remembers positive!)
...
"terrible" → h_final (remembers whole review including "amazing" AND "terrible")

Final output from h_final:
[0.35, 0.65] → 65% negative sentiment

Why?
RNN remembers that despite "amazing", the ending was "terrible"
Which made overall negative
```

### 4. Time Series Prediction

```
Task: Predict next value in sequence

Stock prices: [100, 102, 101, 103, 105, ?]

Process:
100 → h₁
102 → h₂ (remembers 100, 102)
101 → h₃ (remembers trend)
103 → h₄ (remembers upward trend)
105 → h₅ (remembers acceleration)

Predict h₅ → 107 (next price)

Network learns:
- Prices tend upward
- Acceleration pattern
- Seasonal patterns (if trained on long history)
```

### 5. Named Entity Recognition (NER)

```
Task: Label each word with its type

Input: "John works at Google in California"

Process each word:
"John" → PERSON
"works" → VERB
"at" → PREPOSITION
"Google" → ORGANIZATION
"in" → PREPOSITION
"California" → LOCATION

RNN helps because:
"John" is PERSON (context)
"works" is VERB
"Google" after "works at" = ORGANIZATION
Without context, "bank" could be LOCATION or ORGANIZATION
RNN uses context!
```

### 6. Handwriting Generation

```
Task: Generate handwriting stroke by stroke

Input: Sequence of (x, y, pen_pressure)

RNN learns:
How to form each letter based on previous strokes
Creates smooth, realistic handwriting

Can vary style, slant, size by varying input
```

---

## Bidirectional RNN

### Problem with Unidirectional RNN

```
Sentence: "I went to the bank to deposit my ___"

Unidirectional RNN (left to right):
Processes: "I" → "went" → "to" → "the" → "bank" → ...

At "bank": Doesn't know future words "deposit"

Makes decision based on past only
Might predict: "by"

But correct answer is: "check" or "money"
(Because of future context "deposit")
```

### Bidirectional RNN Solution

```
Bidirectional = Read in BOTH directions!

Forward RNN (left to right):
"I" → "went" → "to" → "the" → "bank" → ... 

Backward RNN (right to left):
... "my" → "___" → "deposit" → "to" → "bank" → "the" → ...

Combine both:
Forward sees: "bank" follows "the"
Backward sees: "deposit" follows "bank"

Result at "bank":
Has context from: past AND future
Correct prediction: "check" or "money"!

Shape:
Forward hidden: (32, 128)
Backward hidden: (32, 128)
Combined: (32, 256) [concatenated]

Concatenating doubles the hidden size!
```

### When to Use Bidirectional

```
USE when:
✓ Full sentence available (translation, sentiment)
✓ Not generating text in real-time
✓ Classification task (need full context)

DON'T use when:
✗ Generating next word in real-time
✗ Can't see future (live prediction)
✗ Time series where future unknown
```

---

## Attention Mechanism in RNN

### Problem: Information Loss in Long Sequences

```
Sentence: "The French restaurant owner was angry. The service was slow."

Task: Translate to French

RNN processes all words:
"The" → h₁
"French" → h₂
"restaurant" → h₃
...
"slow" → h_final

Uses only h_final to generate translation!

Problem:
All information compressed into ONE vector
"The" (important) gets lost in final representation
"slow" (at end) dominates

Needs better way to focus on important words!
```

### Attention Solution

```
Instead of using only final hidden state:

Use ALL hidden states!
But "attend to" (focus on) relevant ones

At each decoding step:
1. Calculate importance of each input word
2. Weight them by importance
3. Use weighted sum

Example - Translating "The slow service"

When generating "service" in French:
Attention weights:
"The" → 0.1
"slow" → 0.3
"service" → 0.6 (HIGHEST!)
"was" → 0.0

Weighted combination of all hidden states:
0.1×h₁ + 0.3×h₃ + 0.6×h₅ + 0.0×h₆ + ...

Result: Focus on "service" but remember context!
```

### Attention Formula

```
Simplified:

score_i = similarity(query, key_i)

For each input word:
score = How similar is this word to what we're predicting?

attention_weights = softmax(scores)
(Weights sum to 1, higher = more important)

context = Σ(attention_weight_i × value_i)
(Weighted combination of inputs)

output = combine(context, current_state)
```

### Attention in Practice

```
Encoder-Decoder with Attention:

Encoder RNN:
Processes input sequence
Produces hidden states at each time step

Decoder RNN:
For each output word:
1. Current state: What we're generating
2. Attention: Which input words are relevant?
3. Focus: Weight important input words
4. Decode: Generate output using focus

Result:
"The French restaurant" → Attention focuses on French
                       → Output: "Le restaurant français"

"was slow" → Attention focuses on slow
          → Output: "était lent"
```

---

## Sequence-to-Sequence (Seq2Seq) Architecture

### Complete System

```
ENCODER:
Input: "Hello world"
"Hello" → h₁
"world" → h₂
Final state: c = [0.3, -0.2, 0.5, ...] (captures meaning)

CONTEXT VECTOR (c):
The output of encoder
Contains: Compressed meaning of input sentence

DECODER:
Uses context vector to generate output
START TOKEN → h'₁ + predict → "Bonjour"
"Bonjour" → h'₂ + predict → "le"
"le" → h'₃ + predict → "monde"
END TOKEN → Stop

Output: "Bonjour le monde"
```

### Seq2Seq Use Cases

```
Machine Translation:
Input: English sentence
Output: French sentence

Summarization:
Input: Long article
Output: Short summary

Question Answering:
Input: Question
Output: Answer

Chatbot:
Input: User message
Output: Bot response

Image Captioning:
Input: Image (converted to features by CNN)
Output: Caption sentence
```

---

## Complete RNN Training Example

### Setup

```
Task: Predict next word in reviews
Dataset: 1000 movie reviews
Vocab: 5000 unique words
Sequence length: 20 words per review
Batch size: 32
Hidden size: 128
```

### Epoch 1 Training

```
Batch 1 (32 reviews):
Review 1: "This movie was amazing"
Embed words: [e₁, e₂, e₃, e₄]
Shape: (4, 50) where 50 = embedding size

Feed to RNN:
Step 1: e₁ → h₁
Step 2: e₂ → h₂ (remembers e₁)
Step 3: e₃ → h₃ (remembers e₁, e₂)
Step 4: e₄ → h₄ (remembers e₁, e₂, e₃)

Predict next word: "and" or "but" or "excellent"

Target (actual): "amazing was extremely" (next words)
Calculate loss

Loss₁ = 1.5

Batch 2:
Similar process
Loss₂ = 1.4

...

Epoch 1 Complete:
Average loss: 1.2
```

### Epoch 10 Training

```
Same reviews, but RNN improved

Batch 1:
Process same reviews
But weights updated 9 times already

Loss decreases more:
Loss₁ = 0.3 (much better!)
Loss₂ = 0.28

Average loss: 0.25 (much better than 1.2)
```

### After Training (100 Epochs)

```
Input: "This movie was"
Network predicts: "amazing" (90% confidence)

Input: "The acting was terrible"
Network predicts: "and" (for next word)

Input: "I hated this"
Network predicts: "movie" (negative context)

Network learned patterns:
- "amazing" follows positive phrases
- "movie" follows "this"
- Word dependencies
- Sentiment patterns
```

---

## Common RNN Mistakes and Solutions

### Mistake 1: Not Normalizing Input

```
WRONG:
Raw word embeddings: [100, 200, 150, ...]
Network struggles!

CORRECT:
Normalize embeddings:
[0.1, 0.2, 0.15, ...] (0 to 1 range)

Or use pre-trained embeddings:
Word2Vec, GloVe, FastText
Already normalized
```

### Mistake 2: Sequence Too Long

```
WRONG:
Sequence length: 500 words
Vanishing gradients!
Can't learn dependencies

CORRECT:
Use LSTM/GRU instead of basic RNN
Or limit to 100-200 words
Truncate long sequences
```

### Mistake 3: Forgetting Padding

```
WRONG:
Sequences different lengths:
Sequence 1: 5 words
Sequence 2: 15 words
Can't batch together!

CORRECT:
Pad shorter sequences:
Sequence 1: [word₁, word₂, word₃, word₄, word₅, PAD, PAD, ...]
Sequence 2: [word₁, word₂, ..., word₁₅]

Now both length 15, can batch!
```

### Mistake 4: Using Raw Text

```
WRONG:
Input: "Hello world"
Feed strings directly to RNN!
Network confused!

CORRECT:
1. Tokenize: ["Hello", "world"]
2. Embed: [[0.1, -0.2, 0.5, ...], [0.2, 0.1, -0.3, ...]]
3. Feed embeddings

Numbers, not strings!
```

---

## RNN vs CNN vs Transformer

### RNN (Recurrent)
```
Pros:
✓ Good for sequences
✓ Maintains temporal order
✓ Can handle variable length
✓ Memory of past

Cons:
✗ Slow (sequential processing)
✗ Hard to parallelize
✗ Vanishing gradients
✗ Can't access far-past info easily

Best for:
- Language modeling
- Sequence generation
- Short to medium sequences
```

### CNN (Convolutional)
```
Pros:
✓ Fast (parallel processing)
✓ Good for spatial data
✓ Fewer parameters
✓ Easy to parallelize

Cons:
✗ Doesn't respect sequence order well
✗ Can't access far-away info easily
✗ Designed for 2D data (images)

Best for:
- Images
- Computer vision
- Local pattern detection
```

### Transformer
```
Pros:
✓ Parallel processing (fast!)
✓ Attention to all positions
✓ State-of-the-art NLP
✓ Can handle long sequences

Cons:
✗ More complex
✗ Requires more data
✗ More parameters
✗ Slower for real-time

Best for:
- Large language models (GPT, BERT)
- Machine translation (recent)
- Any sequence with long dependencies
```

---

## Complete Summary - RNN Concepts

### Core Idea

```
RNN processes sequences step by step
Maintains memory of all previous steps
Uses memory to make better predictions

Simple formula:
h_t = activation(W × [h_(t-1), x_t] + b)

h_(t-1) = memory from previous step
x_t = current input
h_t = new memory for next step
```

### Key Variants

```
Basic RNN: Simple, fast, forgets long-term

LSTM: Remembers long-term, slower

GRU: Balance between RNN and LSTM

Bidirectional: Reads both directions

Attention: Focus on important parts
```

### Applications

```
Language:
- Text generation
- Machine translation
- Sentiment analysis
- Named entity recognition

Time:
- Stock prediction
- Weather forecasting
- Traffic prediction

Multimodal:
- Image captioning
- Video understanding
- Speech recognition
```

### Shape Tracking

```
Input: (batch, sequence_length, embedding_size)
       (32, 20, 50)

RNN hidden: (batch, hidden_size)
            (32, 128)

Output: (batch, sequence_length, hidden_size)
        (32, 20, 128)

Final (last step only): (batch, vocab_size)
                        (32, 5000)
```

### Training Tips

```
✓ Use LSTM for long sequences
✓ Normalize embeddings
✓ Pad short sequences
✓ Use gradient clipping
✓ Monitor hidden state for NaN
✓ Start with smaller hidden size
✓ Use pre-trained embeddings if possible
✓ Early stopping to prevent overfitting
```

That's the complete guide to RNN! You now understand the core concept, variants, applications, and how to use them!